# carregando em um dataframe e fazendo testes

In [ ]:
import pandas as pd

nome_arquivo = 'DESTAQUES DE PUBLICAÇÕES.xlsx'

df = pd.read_excel(nome_arquivo, sheet_name='DESTAQUES')

print("--- Primeiras 5 linhas da aba DESTAQUES ---")
display(df.head()) 
print("--- ultimas 5 linhas da aba DESTAQUES ---")
display(df.tail()) 

periodicos_unicos = df['PERIÓDICO'].dropna().unique()

print(f"\nTotal de periódicos únicos para buscar no site: {len(periodicos_unicos)}")
print("\n--- Exemplos dos primeiros periódicos que vamos pesquisar ---")
for p in periodicos_unicos[:5]:
    print(f"- {p}")

--- Primeiras 5 linhas da aba DESTAQUES ---


,ÁREA AVALIAÇÃO,TÍTULO PRODUÇÃO,PERIÓDICO,ISSN,DESTAQUE PARA O DOCENTE,ANO DA PRODUÇÃO,CONCEITO CAPES,ABDC,ABS,JCR,SJR,SPELL
0,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",: WOMEN?S EXPERIENCES IN MANAGEMENT GRADUATE C...,ORGANIZATION (LONDON),1350-5084,MARCELO DE SOUZA BISPO,2024,MB – Muito Bom,A,3.0,Q2,Q1,NaN
1,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",A AMBIDESTRIA ORGANIZACIONAL EM MPES PIAUIENSE...,BASE - REVISTA DE ADMINISTRAÇÃO E CONTABILIDAD...,1984-8196,ALDO LEONARDO CUNHA CALLADO,2024,F – Fraco,NaN,NaN,NaN,NaN,Entre os 40% e 70% melhores
2,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",A GRAPH-THEORETIC APPROACH FOR ASSESSING THE L...,SUPPLY CHAIN MANAGEMENT,1359-8546,CLAUDIA FABIANA GOHR,2023,MB – Muito Bom,A,3.0,Q1,Q1,NaN
3,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",A IMPOSSIBILIDADE DA CIÊNCIA ABERTA SEM ALTERI...,RAC. REVISTA DE ADMINISTRAÇÃO CONTEMPORÂNEA (O...,1982-7849,MARCELO DE SOUZA BISPO,2022,B – Bom,NaN,1.0,NaN,Q3,10% melhores
4,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",ACCRUALS MISPRICING VERSUS RISK: ANALYZING THE...,JOURNAL OF INTERNATIONAL FINANCIAL MANAGEMENT ...,0954-1314,MARCIO ANDRE VERAS MACHADO,2021,MB – Muito Bom,B,2.0,Q1,Q1,NaN


--- ultimas 5 linhas da aba DESTAQUES ---


,ÁREA AVALIAÇÃO,TÍTULO PRODUÇÃO,PERIÓDICO,ISSN,DESTAQUE PARA O DOCENTE,ANO DA PRODUÇÃO,CONCEITO CAPES,ABDC,ABS,JCR,SJR,SPELL
66,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",THE OVER-CONCENTRATION OF INNOVATION AND FIRM-...,JOURNAL OF THE KNOWLEDGE ECONOMY (ONLINE),1868-7873,PEDRO JACOME DE MOURA JUNIOR,2024,B – Bom,C,1.0,NaN,NaN,NaN
67,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",THE OVER-CONCENTRATION OF INNOVATION AND FIRM-...,JOURNAL OF THE KNOWLEDGE ECONOMY (ONLINE),1868-7873,CARLO GABRIEL PORTO BELLINI,2024,B – Bom,C,1.0,NaN,NaN,NaN
68,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",TRABALHO INSTITUCIONAL IDENTITÁRIO NO CAMPO OR...,REVISTA BRASILEIRA DE PESQUISA EM TURISMO,1982-6125,SAMIR ADAMOGLU DE OLIVEIRA,2022,B – Bom,NaN,NaN,Q4,Q4,10% melhores
69,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",TRADUZINDO IDEIAS DE GESTÃO: CONSULTORES COMO ...,CADERNOS EBAPE. BR,1679-3951,SAMIR ADAMOGLU DE OLIVEIRA,2023,B – Bom,NaN,NaN,NaN,NaN,10% melhores
70,"ADMINISTRAÇÃO PÚBLICA E DE EMPRESAS, CIÊNCIAS ...",VULNERABILIDADE NO BEM-ESTAR SOCIAL E NO CONSU...,GESTÃO & REGIONALIDADE (ONLINE),2176-5308,NELSIO RODRIGUES DE ABREU,2024,F – Fraco,NaN,NaN,NaN,NaN,Entre os 40% e 70% melhores



Total de periódicos únicos para buscar no site: 52

--- Exemplos dos primeiros periódicos que vamos pesquisar ---
- ORGANIZATION (LONDON)
- BASE - REVISTA DE ADMINISTRAÇÃO E CONTABILIDADE DA UNISINOS
- SUPPLY CHAIN MANAGEMENT
- RAC. REVISTA DE ADMINISTRAÇÃO CONTEMPORÂNEA (ONLINE)
- JOURNAL OF INTERNATIONAL FINANCIAL MANAGEMENT AND ACCOUNTING (PRINT)


# Testando as requisições ao site

In [ ]:
import requests
from bs4 import BeautifulSoup


periodico_teste = periodicos_unicos[0] 
print(f"Iniciando teste de busca para: {periodico_teste}\n")

url = "https://periodicos-adm.com/"

parametros = {'search_term': periodico_teste}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(url, params=parametros, headers=headers)

if response.status_code == 200:
    print("✅ Conexão com o site bem-sucedida!")
    

    soup = BeautifulSoup(response.text, 'html.parser')
   
    resultados = soup.find_all('div', class_='journal-card')
    
    if resultados:
        print(f"✅ Encontramos {len(resultados)} cartão(ões) de resultado para este periódico!\n")
        print("--- Estrutura HTML extraída do site ---")

        print(resultados[0].prettify()[:800]) 
    else:
        print("")
else:
    print(f"")

Iniciando teste de busca para: ORGANIZATION (LONDON)

✅ Conexão com o site bem-sucedida!



# iniciando a limpeza via re

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import spacy
import pandas as pd


try:
    nlp = spacy.load("pt_core_news_sm")
except:
    import os
    os.system("python -m spacy download pt_core_news_sm")
    nlp = spacy.load("pt_core_news_sm")

# --- MÓDULO DE PROCESSAMENTO (NLP + RE) ---

def processar_periodico_nlp(nome_original):
    """
    Combina Expressões Regulares (Etapa 1) com SpaCy (Etapa 1) 
    para preparar o dado para o Scraping (Etapa 2).
    """
    # [RE] Limpeza inicial: Remove parênteses e ruídos
    nome_limpo_re = re.sub(r'\(.*?\)', '', str(nome_original)).strip()
    
    # [SpaCy] Análise Linguística Profunda
    doc = nlp(nome_limpo_re)
    
    # Extraindo Lemas (radicais) e filtrando apenas o que importa (Substantivos e Adjetivos)
    # Removemos pontuação e stopwords (o, de, com, and, of)
    tokens_interessantes = [
        token.lemma_.lower() for token in doc 
        if not token.is_stop and not token.is_punct and token.pos_ in ['NOUN', 'ADJ', 'PROPN']
    ]
    
    # Criamos uma "assinatura temática" do periódico
    assinatura = " ".join(tokens_interessantes)
    
    return nome_limpo_re, assinatura

# --- EXECUÇÃO E SCRAPING 
# Exemplo pegando o primeiro da sua lista
periodico_raw = periodicos_unicos[0] 

# Aplicando nossa função de NLP
nome_para_busca, termos_chave = processar_periodico_nlp(periodico_raw)

print(f" Nome Original: {periodico_raw}")
print(f"Nome Limpo (RE): {nome_para_busca}")
print(f"Termos Chave (SpaCy): {termos_chave}\n")

# Configuração da requisição (Web Scraping - Etapa 2)
url = "https://periodicos-adm.com/"
parametros = {'search_term': nome_para_busca} 
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print(f" Iniciando busca no site para: {nome_para_busca}...")
response = requests.get(url, params=parametros, headers=headers)

if response.status_code == 200:
    print(" Conexão com o site bem-sucedida!")
    soup = BeautifulSoup(response.text, 'html.parser')
    resultados = soup.find_all('div', class_='journal-card')
    
    if resultados:
        print(f"Sucesso! Encontramos {len(resultados)} resultado(s)!\n")
        
        # Exemplo de extração de dados não estruturados do primeiro card
        primeiro_resultado = resultados[0]
        titulo_site = primeiro_resultado.find('h2').get_text(strip=True) if primeiro_resultado.find('h2') else "N/A"
        
        print(f"Título encontrado no site: {titulo_site}")
        print("--- Estrutura HTML extraída ---")
        print(primeiro_resultado.prettify()[:500]) 
    else:
        print(f"")
else:
    print(f"")

 Nome Original: ORGANIZATION (LONDON)
Nome Limpo (RE): ORGANIZATION
Termos Chave (SpaCy): organization

 Iniciando busca no site para: ORGANIZATION...
 Conexão com o site bem-sucedida!



In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time


print("1. Carregando relatórios e criando mapeamento de ISSN...")
arquivo_relatorio = 'relatorio_producao_intelectual (1).xlsx'
dicionario_abas = pd.read_excel(arquivo_relatorio, sheet_name=None)
df_relatorio = pd.concat(dicionario_abas.values(), ignore_index=True)

df_relatorio['Título do Periódico'] = df_relatorio['Título do Periódico'].astype(str).str.strip().str.upper()
dicionario_issn = df_relatorio.set_index('Título do Periódico')['ISSN'].to_dict()

print("2. Carregando planilha DESTAQUES e formatando colunas...")
arquivo_destaques = 'DESTAQUES DE PUBLICAÇÕES.xlsx'
df_destaques = pd.read_excel(arquivo_destaques, sheet_name='DESTAQUES')

# Corrigindo o problema dos Warnings do Pandas
colunas_notas = ['CONCEITO CAPES', 'ABDC', 'ABS', 'JCR', 'SJR', 'SPELL']
for col in colunas_notas:
    if col in df_destaques.columns:
        df_destaques[col] = df_destaques[col].astype('object')

# Mapeando o ISSN e criando a coluna definitiva 'ISSN'
df_destaques['NOME_UPPER'] = df_destaques['PERIÓDICO'].astype(str).str.strip().str.upper()
df_destaques['ISSN'] = df_destaques['NOME_UPPER'].map(dicionario_issn)


alvos_busca = df_destaques[['ISSN', 'PERIÓDICO']].drop_duplicates().reset_index(drop=True)
resultados_extraidos = {}

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

print(f"\n3. Iniciando a extração para {len(alvos_busca)} periódicos (BUSCA ESTRITA POR ISSN)...\n")

for i, row in alvos_busca.iterrows():
    issn = row['ISSN']
    nome = row['PERIÓDICO']
    
    if pd.isna(issn):
        print(f"[{i+1}/{len(alvos_busca)}] Buscando: {nome} ... ❌ Sem ISSN mapeado. Pulando.")
        continue
        
    print(f"[{i+1}/{len(alvos_busca)}] Buscando ISSN: {issn} ...", end=" ")
    
    # Sistema de Retry (tenta até 3 vezes se a conexão com a internet oscilar)
    for tentativa in range(3):
        try:
            # Busca Exclusiva por ISSN
            resp_busca = requests.get("https://periodicos-adm.com/", params={'search_term': issn}, headers=headers, timeout=15)
            soup_busca = BeautifulSoup(resp_busca.text, 'html.parser')
            link_tag = soup_busca.find('a', class_='card-link')
            
            # Se achou a página
            if link_tag:
                link_detalhe = link_tag.get('href')
                if link_detalhe.startswith('/'):
                    link_detalhe = "https://periodicos-adm.com" + link_detalhe
                
                resp_det = requests.get(link_detalhe, headers=headers, timeout=15)
                soup_det = BeautifulSoup(resp_det.text, 'html.parser')
                
                dados_revista = {'CONCEITO CAPES': '', 'ABDC': '', 'ABS': '', 'JCR': '', 'SJR': '', 'SPELL': ''}
                
                # Coleta CAPES
                capes_tag = soup_det.find('div', class_='cap-box2')
                if capes_tag and capes_tag.find('strong'):
                    dados_revista['CONCEITO CAPES'] = capes_tag.find('strong').text.strip()
                
                # Coleta os demais
                pares = soup_det.find_all('div', class_='pair')
                for par in pares:
                    label_tag = par.find('span', class_='label')
                    val_tag = par.find('span', class_='value')
                    if label_tag and val_tag:
                        label = label_tag.text.strip()
                        val = val_tag.text.strip()
                        if label in dados_revista:
                            dados_revista[label] = val if val != '-' else ''
                            
                resultados_extraidos[nome] = dados_revista
                print("OK!")
                break # Sai do laço de tentativas pois deu certo
            else:
                print("❌ Link não encontrado no site para este ISSN.")
                break # Sai do laço pois o erro não é de conexão, o site que não tem a revista mesmo
                
        except Exception as e:
            if tentativa < 2:
                time.sleep(2) # Espera 2 segundos antes de tentar de novo caso a internet oscile
            else:
                print(f"Erro de conexão após 3 tentativas.")
                
    time.sleep(1) # Pausa entre os periódicos para não sobrecarregar o site


print("\n4. Preenchendo a planilha com os dados coletados...")

for index, row in df_destaques.iterrows():
    nome_linha = row['PERIÓDICO']
    if nome_linha in resultados_extraidos:
        notas = resultados_extraidos[nome_linha]
        df_destaques.at[index, 'CONCEITO CAPES'] = notas['CONCEITO CAPES']
        df_destaques.at[index, 'ABDC'] = notas['ABDC']
        df_destaques.at[index, 'ABS'] = notas['ABS']
        df_destaques.at[index, 'JCR'] = notas['JCR']
        df_destaques.at[index, 'SJR'] = notas['SJR']
        df_destaques.at[index, 'SPELL'] = notas['SPELL']

# Remove a coluna temporária
df_destaques = df_destaques.drop(columns=['NOME_UPPER'])

# Move a coluna ISSN para ficar visualmente melhor (logo depois de PERIÓDICO)
colunas = list(df_destaques.columns)
if 'ISSN' in colunas:
    colunas.insert(colunas.index('PERIÓDICO') + 1, colunas.pop(colunas.index('ISSN')))
    df_destaques = df_destaques[colunas]

print("5. Salvando o arquivo Excel original...")
with pd.ExcelWriter(arquivo_destaques, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_destaques.to_excel(writer, sheet_name='DESTAQUES', index=False)
    
print("\n🎉 Tarefa finalizada! A planilha foi atualizada APENAS via busca por ISSN, garantindo a integridade dos dados.")

1. Carregando relatórios e criando mapeamento de ISSN...
2. Carregando planilha DESTAQUES e formatando colunas...

3. Iniciando a extração para 52 periódicos (BUSCA ESTRITA POR ISSN)...

[1/52] Buscando ISSN: 1350-5084 ... OK!
[2/52] Buscando ISSN: 1984-8196 ... OK!
[3/52] Buscando ISSN: 1359-8546 ... OK!
[4/52] Buscando ISSN: 1982-7849 ... OK!
[5/52] Buscando ISSN: 0954-1314 ... OK!
[6/52] Buscando ISSN: 1982-2596 ... OK!
[7/52] Buscando ISSN: 1678-6971 ... OK!
[8/52] Buscando ISSN: 1984-9230 ... OK!
[9/52] Buscando ISSN: 2525-5584 ... ❌ Link não encontrado no site para este ISSN.
[10/52] Buscando ISSN: 0368-492X ... OK!
[11/52] Buscando ISSN: 2178-938X ... OK!
[12/52] Buscando ISSN: 2177-5184 ... OK!
[13/52] Buscando ISSN: 2175-8069 ... OK!
[14/52] Buscando ISSN: 1981-982X ... OK!
[15/52] Buscando ISSN: 1984-3925 ... OK!
[16/52] Buscando ISSN: 2177-4986 ... ❌ Link não encontrado no site para este ISSN.
[17/52] Buscando ISSN: 2175-6600 ... ❌ Link não encontrado no site para este ISSN.